In [11]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score,
    roc_auc_score,
    precision_recall_curve,
    auc, average_precision_score
)
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.inspection import DecisionBoundaryDisplay

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# Baseline Imports
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from catboost import CatBoostClassifier, CatBoostRegressor

import torch

from tabpfn import TabPFNClassifier, TabPFNRegressor
#from tabpfn_extensions.post_hoc_ensembles.sklearn_interface import AutoTabPFNClassifier, AutoTabPFNRegressor




In [12]:
files = [r"data/PI_DataSet.txt", r"data/INI_DataSet.txt", r"data/NRTI_DataSet.txt", r"data/NNRTI_DataSet.txt"]

for file in files:
    print(file)
    print(file.split("/")[-1].strip(".txt") + "_results.csv")

data/PI_DataSet.txt
PI_DataSe_results.csv
data/INI_DataSet.txt
INI_DataSe_results.csv
data/NRTI_DataSet.txt
NRTI_DataSe_results.csv
data/NNRTI_DataSet.txt
NNRTI_DataSe_results.csv


In [6]:
test =  pd.read_csv("data/INI_DataSet.txt", sep='\t')

drugs = ["CAB", "RAL", "EVG", "DTG", "BIC"]

for drug in drugs:
    print(drug + ": " , test[drug].nunique())

CAB:  0
RAL:  147
EVG:  138
DTG:  62
BIC:  34


In [18]:
input_file = files[1]
#thresholds defined by the database for the classes of "susceptible", "partly susceptible", and "resistant"
thresholds = [
    [3, 15],  # FPV
    [3, 15],  # ATV
    [3, 15],  # IDV
    [9, 55],  # LPV
    [3, 6],  # NFV
    [3, 15],  # SQV
    [2, 8],  # TPV
    [10, 90],  # DRV
    [5, 25],  # X3TC
    [2, 6],  # ABC
    [3, 15],  # AZT
    [1.5, 3],  # D4T
    [1.5, 3],  # DDI
    [1.5, 3],  # TDF
    [3, 10],  # EFV
    [3, 10],  # NVP
    [3, 10],  # ETR
    [3, 10],  # RPV
    [2.5, 10],  # BIC
    [4, 13],  # DTG
    [2.5, 10],  # EVG - upper threshold guessed
    [1.5, 10]  # RAL - upper threshold guessed
]

# Define row and column names
index = ["FPV", "ATV", "IDV", "LPV", "NFV", "SQV", "TPV", "DRV",
         "3TC", "ABC", "AZT", "D4T", "DDI", "TDF",
         "EFV", "NVP", "ETR", "RPV", "BIC", "DTG", "EVG", "RAL"]
columns = ["lower", "upper"]

# Create DataFrame
cutoff_df = pd.DataFrame(thresholds, index=index, columns=columns)

# Reading in and processing high quality File

df = pd.read_csv(input_file, sep='\t')
#print(df)
df = df.iloc[:,1:-1]
print(df)

#Checking how much data is available for each drug
#print(df.loc[:,"FPV":"DRV"].count())

#list of current drugs of the dataset
drugs = [drug for drug in list(df.columns) if not drug.startswith("P") ]

#creating the one hot encoding for the features
enc = OneHotEncoder(handle_unknown='error')

enc.fit(df.loc[:,[drug for drug in list(df.columns) if drug.startswith("P")]])


#going through the drugs and splitting them to test and training depending on the drug

results = pd.DataFrame(columns=["Drug",
                                "RMSE",
                                "AUC ROC MC",
                                "AUC PRC MC",
                                "AUC ROC BI",
                                "AUC PRC BI",
                                "Time"])

print(input_file)

for drug in drugs:
    #print(drug)
    tmp_drugs = drugs.copy()
    #print(tmp_drugs)
    tmp_drugs.remove(drug)
    #print(tmp_drugs)
    last_col = list(df.columns)[-1]
    dataframe = df.drop(tmp_drugs, axis=1)

    #print(dataframe.head())

    dataframe = dataframe.dropna()

    '''
    # encoding the levels of susceptibility as 0 for susceptible, 1 as resistant
    dataframe.loc[dataframe[drug] < cutoff_df.loc[drug, "lower"], drug + "_level_binary"] = 0
    dataframe.loc[dataframe[drug] >= cutoff_df.loc[drug, "lower"], drug + "_level_binary"] = 1


    # encoding the levels of susceptibility as 0 for susceptible, 1 as partly resistant and 2 as completely resistant
    dataframe.loc[dataframe[drug] < cutoff_df.loc[drug, "lower"], drug + "_level"] = 0
    dataframe.loc[dataframe[drug] >= cutoff_df.loc[drug, "upper"], drug + "_level"] = 2
    dataframe.loc[(dataframe[drug] >= cutoff_df.loc[drug, "lower"]) & (
                dataframe[drug] < cutoff_df.loc[drug, "upper"]), drug + "_level"] = 1
    '''
    #print(dataframe.head())

    X, y = dataframe.drop([drug], axis=1), np.array(dataframe[drug])

    print(y)

    print(X.shape)
    if X.shape[0] == 0:
        continue

    X_trafo = enc.transform(X).toarray()

    #print(X)


      RAL   EVG  DTG  BIC  CAB P1 P2 P3 P4 P5  ... P279 P280 P281 P282 P283  \
0     NaN   4.0  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
1     1.4  14.5  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
2     9.8  33.7  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
3     1.0  37.4  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
4     1.0  93.8  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
..    ...   ...  ...  ...  ... .. .. .. .. ..  ...  ...  ...  ...  ...  ...   
760  11.0   NaN  1.0  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
761   1.1   1.3  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    G   
762   1.2   1.4  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
763   0.8   1.1  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   
764   1.1   1.5  NaN  NaN  NaN  -  -  -  -  -  ...    -    -    -    -    -   

    P284 P285 P286 P287 P288  
0      -    -    -  

In [10]:
x = 9
y = 2

x // y + x % y


5